In [1]:
#| default_exp restxl

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [6]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length,
    #local_files_only=True
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

[W socket.cpp:426] [c10d] The server socket cannot be initialized on [::]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).


> initializing model parallel with size 1


404 Client Error: Not Found for url: https://huggingface.co/sberbank-ai/rugpt3xl/resolve/main/config.json


Use alternating sparse & dense attention layers


In [7]:
sum(p.numel() for p in model.parameters())

1315737600

In [8]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.half,
                                 checkpoint=None,
                                 replace_method='auto',
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2023-02-13 18:24:29,172] [INFO] [logging.py:68:log_dist] [Rank -1] DeepSpeed info: version=0.8.0, git-hash=unknown, git-branch=unknown
[2023-02-13 18:24:29,173] [WARNING] [config_utils.py:67:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2023-02-13 18:24:29,174] [INFO] [logging.py:68:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [9]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [10]:
%%time
get_sample(' - ты кто? \n - ', 50, 4, False)

CPU times: user 20.1 s, sys: 1.04 s, total: 21.1 s
Wall time: 9.13 s


['̀Разрываюсь я, Разрываюся на части, Думала на все согласна, А любовь-то не велит. - Что ж не хочешь? ˗ Мне ведь отдашься, Дуреха, легко?',
 'Я прЫнцесса из страны Рыбной. Назовите ваш адрес, я приду, проведаю вас. А вас как звать? - Давайте, просто: Иван, послушайте меня, уйдите с дороги!',
 ' я-гитарист, я за все в ответе. А, ну это просто такая гитарка, на которой надо только играть. Ничего больше! Ну, знаешь, как этим девочкам на концертах музыканты делают?',
 'ыыыхын! Я лысый васхын-ая-айын!" - и черные вороны полетели вниз.']

In [11]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

 Дрозд? Гриф? Беркут? Гром?
Всех я видел зёрен незримых слитки.
Ты вот выпьешь литр молочной воды -
И будешь весь мой, весь в моей крови!

CPU times: user 49.4 s, sys: 1.97 s, total: 51.4 s
Wall time: 2.04 s
